In [1]:
"""
Callset benchmarking and comparison.
"""

import copy
import importlib
import re
import pickle
import warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib as mpl
import pyranges as pr
import pysam
from scipy.spatial import cKDTree
import os
from typing import Union, Tuple, Optional, List, Any, Literal, Dict


workdir = "/vast/ai_projects/Papenfuss_SVEnsemble2"
os.chdir(workdir)
os.getcwd()

import svensemble.vcfparse as vcfp # noqa: E402 (flake8)
import svensemble.breakends as bnds
import svensemble.breakpoints as bps
import svensemble.events as evts
import svensemble.caller_specific as caller_specific
import svensemble.benchmark as benchmark
import svensemble._caller_params as caller_params
import svensemble.chunk_match as chunk_match
import svensemble.matched_breakpoints as matchedbps
import svensemble.utils as sv_utils

importlib.reload(vcfp)
importlib.reload(bnds)
importlib.reload(bps)
importlib.reload(evts)
importlib.reload(caller_specific)
importlib.reload(benchmark)
importlib.reload(caller_params)
importlib.reload(chunk_match)
importlib.reload(matchedbps)
importlib.reload(sv_utils)

pd.set_option('display.max_columns', None)

human_chroms = [f'chr{i}' for i in range(1, 23)] + ['chrX', 'chrY', 'chrM']

long_val = True


In [2]:
pu_savedir = Path('saved/pu')
pu_savedir.mkdir(parents
                 =True, exist_ok=True)

joint_bp_df_path = Path('saved/matched/joint_bp_df.csv')

training_sets =  {
    'gridss': ['gridss'],
    'manta_candidate': ['manta_candidate'],
    'manta_diploid': ['manta_diploid'],
    'gridss+manta_candidate': ['gridss', 'manta_candidate'],
    'gridss+manta_diploid': ['gridss', 'manta_diploid']
}
sample_names = ['NA12878_30x', 'CHM13', 'NA12878', 'HG002']
# sample_names = ['NA12878_30x']

svens2_model_name = 'unbiased'

long_suffix = '_long' if long_val else ''

sample_val_sets = {
    'NA12878': {
        'name': 'kosugi',
        'path': Path(f'data/Kosugi_validation/NA12878_DGV-2016_LR-assembly.hg38.{long_suffix}.bps.csv')
    },
    'NA12878_30x': {
        'name': 'kosugi',
        'path': Path(f'data/Kosugi_validation/NA12878_DGV-2016_LR-assembly.hg38.{long_suffix}.bps.csv')
    },
    'CHM13': {
        'name': 'chm13_val',
        'path': Path(f'data/CHM13_validation/supporting_variants_for_nstd137.{long_suffix}.bps.csv')
    },
    'HG002': {
        'name': 'hg002_val',
        'path': Path(f'data/HG002_validation/HG002_validation.{long_suffix}.bps.csv')
    }
}
callset_qual_metrics = {
    'gridss': 'QUAL',
    'manta_candidate': 'INFO_TOTAL_PAIR_COUNT',
    'manta_diploid': 'QUAL'
}

samples = {}
for sample_name in sample_names:
    samples.update(sv_utils.create_config_dict(sample_name))
    samples[sample_name]['val_set'] = sample_val_sets[sample_name]

In [3]:
if not Path(sample_val_sets['HG002']['path']).exists():

    hg002_val_vcf = pysam.VariantFile("/home/users/allstaff/shikov.v/projects/Papenfuss_SVEnsemble2/data/HG002_validation/HG002_validation.vcf.gz")
    hg002_val_vcf_df, hg002_val_vcf_meta = vcfp.parse_vcf_to_df(hg002_val_vcf, caller_params=caller_params.hg002_params)

    hg002_val_vcf_df = hg002_val_vcf_df[hg002_val_vcf_df['INFO_SVTYPE'].notna()]

    hg002_val_vcf_df.loc[:, 'INFO_EVENT'] = ['HG002_' + str(i) for i in range(len(hg002_val_vcf_df))]
    hg002_val_vcf_df.loc[:, 'ID'] = hg002_val_vcf_df['INFO_EVENT']

    hg002_val_vcf_df.loc[:, 'INFO_CIPOS_l'] = -10
    hg002_val_vcf_df.loc[:, 'INFO_CIPOS_r'] = 10
    hg002_val_vcf_df.loc[:, 'INFO_CIEND_l'] = -10
    hg002_val_vcf_df.loc[:, 'INFO_CIEND_r'] = 10
    hg002_val_vcf_df.loc[:, 'INFO_CHR2'] = hg002_val_vcf_df['CHROM']

    hg002_val_vcf_df_bnds = evts.event_to_bnd(hg002_val_vcf_df)
    hg002_val_vcf_df_bps = bnds.bnd_to_bp(hg002_val_vcf_df_bnds)

    if long_val:
        hg002_val_vcf_df_bps = hg002_val_vcf_df_bps[hg002_val_vcf_df_bps['INFO_SVLEN'] > 50]

    hg002_val_vcf_df_bps.to_csv(sample_val_sets['HG002']['path'], index=False)

In [4]:
if not Path(sample_val_sets['NA12878']['path']).exists():

    val_df = benchmark.load_truth_variants('data/Kosugi_validation/NA12878_DGV-2016_LR-assembly.hg38.bed')
    val_vcf = benchmark.truth_to_vcf_df(val_df)

    val_vcf_bps = benchmark.truth_event_to_bnd(val_vcf)
    if long_val:
        val_vcf_bps = val_vcf_bps[val_vcf_bps['INFO_SVLEN'] > 50]
    val_vcf_bps.to_csv(sample_val_sets['NA12878']['path'], index=False)

In [5]:
if not Path(sample_val_sets['CHM13']['path']).exists():

    chm_val_df = pd.read_csv('data/CHM13_validation/supporting_variants_for_nstd137.csv')

    chm_val_vcf_df = chm_val_df.copy()

    chm_val_vcf_df = chm_val_vcf_df.dropna(subset=['Start', 'End'])
    chm_val_vcf_df = chm_val_vcf_df[chm_val_vcf_df['Assembly Name'] == 'GRCh38 (hg38)']

    chm_rename_dict = {
        'insertion': 'INS',
        'deletion': 'DEL',
        'inversion': 'INV'
    }

    chm_val_vcf_df.loc[:, 'Variant Call type'] = chm_val_vcf_df['Variant Call type'].map(chm_rename_dict)

    chm_val_vcf_df.loc[:, 'Start'] = chm_val_vcf_df['Start'].astype(int)
    chm_val_vcf_df.loc[:, 'End'] = chm_val_vcf_df['End'].astype(int)

    chm_val_cols_to_keep = ['CHROM', 'POS', 'INFO_CHR2', 'REF', 'ALT', 'ID', 'INFO_EVENT', 'INFO_SVTYPE', 'INFO_SVLEN', 'INFO_END', 'INFO_CIPOS_l', 'INFO_CIPOS_r', 'INFO_CIEND_l', 'INFO_CIEND_r']

    chm_val_vcf_df.loc[:, 'CHROM'] = 'chr' + chm_val_vcf_df['Chromosome'].astype(str)
    chm_val_vcf_df.loc[:, 'POS'] = chm_val_vcf_df['Start'] + 1
    chm_val_vcf_df.loc[:, 'INFO_CHR2'] = 'chr' + chm_val_vcf_df['Chromosome'].astype(str)
    chm_val_vcf_df.loc[:, 'INFO_END'] = chm_val_vcf_df['End']
    chm_val_vcf_df.loc[:, 'REF'] = '.'
    chm_val_vcf_df.loc[:, 'ALT'] = '.'
    chm_val_vcf_df.loc[:, 'ID'] = chm_val_vcf_df['Variant ID']
    chm_val_vcf_df.loc[:, 'INFO_EVENT'] = chm_val_vcf_df['Variant ID']
    chm_val_vcf_df.loc[:, 'INFO_SVTYPE'] = chm_val_vcf_df['Variant Call type']
    chm_val_vcf_df.loc[:, 'INFO_SVLEN'] = 0
    chm_val_vcf_df.loc[:, 'INFO_CIPOS_l'] = -10
    chm_val_vcf_df.loc[:, 'INFO_CIPOS_r'] = 10
    chm_val_vcf_df.loc[:, 'INFO_CIEND_l'] = -10
    chm_val_vcf_df.loc[:, 'INFO_CIEND_r'] = 10

    chm_val_vcf_df = chm_val_vcf_df[chm_val_cols_to_keep]
    chm_val_vcf_df = chm_val_vcf_df[chm_val_vcf_df['CHROM'].isin(human_chroms)]

    bnds.bnd_to_bp(evts.event_to_bnd(chm_val_vcf_df)).to_csv(sample_val_sets['CHM13']['path'], index=False)

In [6]:
for sample_name in samples.keys():
    samples[sample_name].update({'matched_bps': {training_set_name: None for training_set_name in samples[sample_name]['training_sets']}})
    for training_set_name in samples[sample_name]['training_sets']:
        samples[sample_name]['matched_bps'][training_set_name] = pd.read_csv(samples[sample_name]['sample_dir'] / 'sv_calling' / 'svensemble2' / f'matched_bps_{training_set_name}.csv', low_memory=False)
    samples[sample_name]['pred_dicts'] = pickle.load(open(samples[sample_name]['sample_dir'] / 'sv_calling' / 'svensemble2' / 'unbiased_dict.pkl', 'rb'))
    # Predictions have ID columns as index, change them to usual columns
    for training_set, pred_dict in samples[sample_name]['pred_dicts'].items():
        pred_dict['y'].reset_index(inplace=True, drop=False)
        pred_dict['y'].drop(columns=['gnomad_INFO_EVENT', 0], inplace=True)
        pred_dict['y'].rename(columns={1: f'{training_set}_{svens2_model_name}_pred'}, inplace=True)
    # Ensure all column names in pred_dicts prediction dataframes are strings, not integers
    # for training_set, pred_dict in samples[sample_name]['pred_dicts'].items():
    #     pred_dict['y'].columns = pred_dict['y'].columns.astype(str)


In [7]:
# def merge_predictions(matched_bps: pd.DataFrame, preds: pd.DataFrame, training_set_name: str, svens2_model_name: str):
#     # First, check if matched_bps contains INFO_EVENT columns from preds
#     matched_bps_info_event_cols = [col for col in matched_bps.columns if 'INFO_EVENT' in col]
#     pred_info_event_cols = [col for col in preds.columns if 'INFO_EVENT' in col]
#     if set(matched_bps_info_event_cols).intersection(set(pred_info_event_cols)) != set(pred_info_event_cols):
#         raise ValueError(f"INFO_EVENT columns in matched_bps and pred_dict do not match: {matched_bps_info_event_cols} != {pred_info_event_cols}, cannot create matches")

#     # Pick the column with svensemble2 positive predictions from preds and rename it
#     # The required column is always the last one
#     svens2_pred_col_name = f"{training_set_name}_{svens2_model_name}_pred"
#     preds = preds[pred_info_event_cols + [preds.columns[-1]]]
#     preds = preds.rename({preds.columns[-1]: svens2_pred_col_name}, axis=1)

#     # Left-merge preds with matched_bps on the INFO_EVENT columns
#     matched_bps = matched_bps.merge(preds, on=pred_info_event_cols, how='left')
#     return matched_bps

In [8]:
for sample_name in samples.keys():
    for training_set_name, training_set_callsets in samples[sample_name]['training_sets'].items():
        if 'gridss' in training_set_callsets:
            samples[sample_name]['matched_bps'][training_set_name].loc[:, 'gridss_FILTER_PASS'] = np.all(samples[sample_name]['matched_bps'][training_set_name][['gridss_FILTER_PASS_bnd1', 'gridss_FILTER_PASS_bnd2']].astype(bool), axis=1)
        if 'manta_candidate' in training_set_callsets:
            samples[sample_name]['matched_bps'][training_set_name].loc[:, 'manta_candidate_INFO_TOTAL_PAIR_COUNT'] = \
                samples[sample_name]['matched_bps'][training_set_name]['manta_candidate_INFO_UPSTREAM_PAIR_COUNT'] + \
                samples[sample_name]['matched_bps'][training_set_name]['manta_candidate_INFO_DOWNSTREAM_PAIR_COUNT'] + \
                samples[sample_name]['matched_bps'][training_set_name]['manta_candidate_INFO_PAIR_COUNT']


In [9]:
# Set '[training_set]_unbiased_pred' column value for matched breakpoints to the predicted value. Put NA if there is no prediction

for sample_name in samples:
    for training_set, training_callsets in training_sets.items():
        model_name = 'unbiased'

        samples[sample_name]['matched_bps'][training_set] = samples[sample_name]['matched_bps'][training_set].merge(
            samples[sample_name]['pred_dicts'][training_set]['y'],
            on=[f'{callset}_INFO_EVENT' for callset in training_callsets]
        )


In [ ]:
for sample_name, sample in samples.items():
    samples[sample_name].update({'svensemble2_calls': {}})
    for training_set_name, training_callsets in sample['training_sets'].items():
        matched_bps = sample['matched_bps'][training_set_name].copy()
        matched_bps['svensemble2_insert_size'] = matchedbps.get_combined_insert_size(matched_bps, training_callsets)
        #create a unique id for each sv
        svensemble2_id = pd.Series(index=matched_bps.index, dtype=str)
        svensemble2_id = matched_bps[[f'{callset}_INFO_EVENT' for callset in training_callsets]].astype(str).agg('|'.join, axis=1)
        svensemble2_id = 'svensemble2_' + pd.factorize(svensemble2_id)[0].astype(str)
        matched_bps.loc[:, 'INFO_EVENT'] = svensemble2_id

        samples[sample_name]['svensemble2_calls'].update(
            {
                training_set_name: matchedbps.matchedbps_to_bps(matched_bps, training_callsets, extra_cols=['svensemble2_insert_size', 'INFO_EVENT'] + [f'{training_set_name}_{model_name}_pred'])
            }
        )


In [ ]:
verbose = True

# Here we create a validation dictinary containing
# Validation dictionary is a nested dictionary, with upper level keys being the sample names
# For each sample, a dictionary exists with for each callset as well as for each svensemble2 prediction

val_dict = defaultdict(dict)

for sample_name, sample in samples.items():
    if sample['val_set'] is None:
        continue

    val_set = pd.read_csv(sample['val_set']['path'])

    # First, handle original callsets that will be compared to svensemble2 predictions
    for callset in sample['callsets']:
        if verbose:
            print(f'Processing {callset} callset for {sample_name}')

        # Recreate the original callset bps df by picking only the rows that contain the corresponding INFO_EVENT
        # Then remove all columns without the callset prefix
        callset_bps = pd.read_csv(sample['callsets'][callset]['svensemble_dir'] / f'{callset}_bps.csv', low_memory=False)

        # Caller-specific operations go here
        if callset == 'manta_candidate':
            callset_bps.loc[:, 'INFO_TOTAL_PAIR_COUNT'] = \
                callset_bps['INFO_UPSTREAM_PAIR_COUNT'] + \
                callset_bps['INFO_DOWNSTREAM_PAIR_COUNT'] + \
                callset_bps['INFO_PAIR_COUNT']

        matched_ids = chunk_match.inner_join_bps(
            bp_df_dict={
                callset: callset_bps,
                sample['val_set']['name']: val_set
            },
            max_dist=200,
            ids_only=True,
            ignore_size=True
        )
        callset_bps.loc[:, 'is_in_val'] = callset_bps['INFO_EVENT'].isin(matched_ids[f'{callset}_INFO_EVENT'])
        cols_to_keep = ['INFO_EVENT', f'{callset_qual_metrics[callset]}', 'is_in_val']
        if 'FILTER_PASS' in callset_bps.columns:
            cols_to_keep.append('FILTER_PASS')
        val_dict[sample_name][callset] = callset_bps.loc[callset_bps['INFO_EVENT'].notna(), cols_to_keep].copy()

        if verbose:
            match_count = val_dict[sample_name][callset]['is_in_val'].sum()
            print(f'{callset} callset for {sample_name} has {match_count} rows matching the validation set')

    # Then, handle svensemble2 predictions

    for training_set, training_callsets in samples[sample_name]['training_sets'].items():

        if verbose:
            print(f'Processing SVEnsemble2 predictions with {training_set} for {sample_name}')

        svens_calls = sample['svensemble2_calls'][training_set]

        matched_ids = chunk_match.inner_join_bps(
            bp_df_dict={
                'svensemble2': svens_calls,
                sample['val_set']['name']: val_set
            },
            max_dist=200,
            ids_only=True,
            ignore_size=True
        )

        svens_calls.loc[:, 'is_in_val'] = svens_calls['INFO_EVENT'].isin(matched_ids['svensemble2_INFO_EVENT'])

        val_df = svens_calls[
            [
                'INFO_EVENT',
                f'{training_set}_{model_name}_pred',
                'is_in_val'
            ]
        ]
        val_df = val_df[val_df[f'{training_set}_{model_name}_pred'].notna()]

        val_dict[sample_name][f'svensemble2_{training_set}'] = val_df.copy()

        if verbose:
            match_count = val_dict[sample_name][f'svensemble2_{training_set}']['is_in_val'].sum()
            print(f'SVEnsemble2  predictions with {training_set} for {sample_name} has {match_count} rows matching the validation set')


In [12]:
pickle.dump(val_dict, open('val_dict.pkl', 'wb'))

In [ ]:
val_dict = pickle.load(open('val_dict.pkl', 'rb'))

In [ ]:
for sample_name in val_dict.keys():
    for pred_set in val_dict[sample_name].keys():
        print(f'Sample: {sample_name},\nPred set: {pred_set},\nCounts: {val_dict[sample_name][pred_set]["is_in_val"].value_counts()}\n\n')

In [6]:
from sklearn.metrics import precision_recall_curve
from matplotlib import pyplot as plt
from sklearn.metrics import roc_curve


In [ ]:
val_dict['NA12878_30x']['manta_candidate'][val_dict['NA12878_30x']['manta_candidate']['INFO_TOTAL_PAIR_COUNT'] != 0]

In [ ]:
test = val_dict['NA12878_30x']['manta_candidate']
roc_curve(test['is_in_val'], test.iloc[:, 1])

In [ ]:
plot_reverse_roc_curves(val_dict['NA12878_30x'], filter_pass=False, title='NA12878_30x')

In [ ]:
plot_roc_curves(val_dict['NA12878_30x'], filter_pass=False, title='NA12878_30x')

In [ ]:
plot_roc_curves({val_set_name: val_set for val_set_name, val_set in val_dict['NA12878_30x'].items() if val_set_name in ['manta_diploid', 'svensemble2_manta_diploid']}, filter_pass=False, title='NA12878_30x')

In [ ]:
plot_roc_curves(val_dict['NA12878'], filter_pass=True, title='NA12878')

In [ ]:
plot_roc_curves(val_dict['CHM13'], filter_pass=True, title='CHM13')

In [ ]:
plot_roc_curves(val_dict['HG002'], ax=None, use_counts=True, title='HG002')

In [ ]:
plot_precision_recall_curves(val_dict['HG002'])